In [1]:
# ================================================================
# PM2.5 Level Prediction using CNN (All Features)
# ================================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm

# ---------------- 1) Load dataset ----------------
train_df = pd.read_csv("train_split.csv")
test_df = pd.read_csv("test_split.csv")

X_train = train_df.drop(columns=["PM2.5_level"]).values.astype(np.float32)
y_train = train_df["PM2.5_level"].values.astype(np.int64) - 1  # shift to 0-based

X_test = test_df.drop(columns=["PM2.5_level"]).values.astype(np.float32)
y_test = test_df["PM2.5_level"].values.astype(np.int64) - 1

num_classes = len(np.unique(y_train))
input_dim = X_train.shape[1]

# ---------------- 2) Scale data ----------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# reshape for CNN -> (samples, 1, features)
X_train = X_train.reshape(-1, 1, input_dim)
X_test = X_test.reshape(-1, 1, input_dim)

# ---------------- 3) Define CNN model ----------------
class CNNModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.dropout = nn.Dropout(0.4)
        self.fc1 = nn.Linear(64 * input_dim, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# ---------------- 4) K-Fold Validation ----------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
val_metrics = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    print(f"\nFold {fold}...")
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    train_ds = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
    val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)

    model = CNNModel(input_dim, num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, verbose=False)

    best_val_loss = np.inf
    patience, wait = 10, 0

    for epoch in range(50):
        model.train()
        total_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # validation
        model.eval()
        val_preds, val_probs, val_true = [], [], []
        val_loss = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                loss = criterion(out, yb)
                val_loss += loss.item()
                probs = F.softmax(out, dim=1).cpu().numpy()
                preds = np.argmax(probs, axis=1)
                val_probs.extend(probs)
                val_preds.extend(preds)
                val_true.extend(yb.cpu().numpy())

        val_loss /= len(val_loader)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = model.state_dict().copy()
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_weights)

    acc = accuracy_score(val_true, val_preds)
    prec = precision_score(val_true, val_preds, average='weighted', zero_division=0)
    rec = recall_score(val_true, val_preds, average='weighted', zero_division=0)
    f1 = f1_score(val_true, val_preds, average='weighted', zero_division=0)
    y_bin = label_binarize(val_true, classes=np.arange(num_classes))
    val_probs = np.array(val_probs)
    try:
        roc_auc = roc_auc_score(y_bin, val_probs, average='weighted', multi_class='ovr')
    except:
        roc_auc = np.nan

    val_metrics.append({
        "Fold": fold,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-score": f1,
        "ROC-AUC": roc_auc
    })

validation_table = pd.DataFrame(val_metrics)
print("\nValidation Results:")
print(validation_table)



Fold 1...


G:\anaconda\envs\Senior\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(



Fold 2...


G:\anaconda\envs\Senior\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(



Fold 3...


...


Fold 4...


G:\anaconda\envs\Senior\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(



Fold 5...


G:\anaconda\envs\Senior\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(



Validation Results:
   Fold  Accuracy  Precision    Recall  F1-score   ROC-AUC
0     1  0.776595   0.767089  0.776595  0.769756  0.958032
1     2  0.773773   0.767917  0.773773  0.767413  0.957801
2     3  0.705332   0.705597  0.705332  0.698750  0.939345
3     4  0.776212   0.768810  0.776212  0.769664  0.958519
4     5  0.776349   0.764145  0.776349  0.767195  0.958334


In [4]:
# ---------------- 5) Train on full train set (with metrics log) ----------------
train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)

final_model = CNNModel(input_dim, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=1e-3, weight_decay=1e-4)

train_history = []  # เก็บผลลัพธ์แต่ละ epoch

for epoch in tqdm(range(30), desc="Training full model"):
    final_model.train()
    total_loss = 0
    all_preds, all_probs, all_true = [], [], []

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = final_model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        # บันทึก prediction สำหรับคำนวณ metric ระหว่างเทรน
        probs = F.softmax(out, dim=1).detach().cpu().numpy()
        preds = np.argmax(probs, axis=1)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_true.extend(yb.cpu().numpy())

    # ----- คำนวณ metrics ต่อ epoch -----
    avg_loss = total_loss / len(train_loader)
    acc = accuracy_score(all_true, all_preds)
    prec = precision_score(all_true, all_preds, average='weighted', zero_division=0)
    rec = recall_score(all_true, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_true, all_preds, average='weighted', zero_division=0)
    y_bin = label_binarize(all_true, classes=np.arange(num_classes))
    try:
        roc_auc = roc_auc_score(y_bin, np.array(all_probs), average='weighted', multi_class='ovr')
    except:
        roc_auc = np.nan

    train_history.append({
        "Epoch": epoch + 1,
        "Loss": avg_loss,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-score": f1,
        "ROC-AUC": roc_auc
    })

# แปลงเป็น DataFrame
train_progress_table = pd.DataFrame(train_history)
print("\nTraining Progress:")
print(train_progress_table.tail())  # แสดงท้ายตาราง 5 แถวสุดท้าย


Training full model: 100%|██████████| 30/30 [04:06<00:00,  8.20s/it]


Training Progress:
    Epoch      Loss  Accuracy  Precision    Recall  F1-score   ROC-AUC
25     26  0.578981  0.760903   0.747549  0.760903  0.751092  0.951752
26     27  0.578815  0.761318   0.748286  0.761318  0.751602  0.951788
27     28  0.578722  0.761069   0.747813  0.761069  0.751300  0.951789
28     29  0.578203  0.761485   0.748280  0.761485  0.751868  0.951877
29     30  0.577202  0.761992   0.748874  0.761992  0.752352  0.952073


In [5]:
# ---------------- 6) Test Evaluation ----------------
final_model.eval()
with torch.no_grad():
    X_t = torch.tensor(X_test).to(device)
    out = final_model(X_t)
    y_test_proba = F.softmax(out, dim=1).cpu().numpy()
    y_test_pred = np.argmax(y_test_proba, axis=1)

acc_t = accuracy_score(y_test, y_test_pred)
prec_t = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
rec_t = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
f1_t = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)
y_test_bin = label_binarize(y_test, classes=np.arange(num_classes))
try:
    roc_auc_t = roc_auc_score(y_test_bin, y_test_proba, average='weighted', multi_class='ovr')
except:
    roc_auc_t = np.nan

test_table = pd.DataFrame([{
    "Accuracy": acc_t,
    "Precision": prec_t,
    "Recall": rec_t,
    "F1-score": f1_t,
    "ROC-AUC": roc_auc_t
}])

print("\nTest Results:")
print(test_table)


Test Results:
   Accuracy  Precision    Recall  F1-score   ROC-AUC
0  0.763027   0.760046  0.763027  0.758182  0.953898


In [6]:
# ---------------- 7) Save Results ----------------
validation_table.to_csv("Baseline_validation_metrics_cnn.csv", index=False)
test_table.to_csv("Baseline_test_smetrics_cnn.csv", index=False)

proba_df = pd.DataFrame(y_test_proba, columns=[f"prob_{c}" for c in np.unique(y_train)])
proba_df.insert(0, "true", y_test)
proba_df.to_csv("Baseline_proba_test_cnn.csv", index=False)

print("\nSaved: BS_allclass_validation_metrics_cnn.csv, BS_allclass_test_metrics_cnn.csv, BS_allclass_proba_test_cnn.csv")


Saved: BS_allclass_validation_metrics_cnn.csv, BS_allclass_test_metrics_cnn.csv, BS_allclass_proba_test_cnn.csv
